In [11]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
#qui creo un vettore per ogni paziente

base_dir = r"C:\Users\mediaw\Downloads\AverageValues_Voxelwise"

from sklearn.preprocessing import MinMaxScaler

# Definisci la lista delle colonne da normalizzare
columns_to_normalize = ['Mua1', 'Mua2', 'Mua3', 'Mua4', 'Mua5', 'Mua6', 'Mua7', 'Mua8', 
                        'Hb', 'HbO2', 'Lipid', 'H2O', 'Collagen', 'HbTot', 'SO2']

# Inizializza lo scaler
scaler = MinMaxScaler()

# Dizionario per memorizzare i vettori di medie per ciascun file di testo
mean_vectors_by_file = {}


# Itera su tutti i file di testo
for file_name in os.listdir(base_dir):
    # Verifica che il file sia di testo
    if file_name.endswith('.txt'):
        # Leggi il file di testo e crea un DataFrame
        file_path = os.path.join(base_dir, file_name)
        df = pd.read_table(file_path)
        
        # Seleziona solo le colonne da normalizzare
        df_normalized = df[columns_to_normalize].copy()
        
        # Normalizza le colonne selezionate
        df_normalized = pd.DataFrame(scaler.fit_transform(df_normalized), columns=columns_to_normalize)
        
        # Calcola le medie delle colonne
        column_means = df_normalized.mean()
        
        # Aggiungi le medie delle colonne al dizionario
        mean_vectors_by_file[file_name] = column_means.values

# Stampare il numero di vettori di medie ottenuti
print("Numero di vettori di medie ottenuti:", len(mean_vectors_by_file))

# Ora il dizionario mean_vectors_by_file contiene i vettori di medie per ciascun file di testo
#In questo codice, mean_vectors_by_file è un dizionario in cui le chiavi sono i nomi dei file di testo e i valori sono i vettori di medie corrispondenti. 
#Alla fine del processo, il dizionario conterrà 33 coppie di chiavi-valore, ognuna rappresentante il nome del file di testo e il vettore di medie corrispondente.

Numero di vettori di medie ottenuti: 33


In [12]:
#qui creo quattro vettori per ogni paziente a seconda del valore di combination

base_dir = r"C:\Users\mediaw\Downloads\AverageValues_Voxelwise"
from sklearn.preprocessing import MinMaxScaler

# Definisci la lista delle colonne da normalizzare
columns_to_normalize = ['Mua1', 'Mua2', 'Mua3', 'Mua4', 'Mua5', 'Mua6', 'Mua7', 'Mua8', 
                        'Hb', 'HbO2', 'Lipid', 'H2O', 'Collagen', 'HbTot', 'SO2']

# Inizializza lo scaler
scaler = MinMaxScaler()

# Dizionario per memorizzare i vettori di medie per ciascun file di testo e per ciascun valore di combination
mean_vectors_by_file_and_combination = {}


# Itera su tutti i file di testo
for file_name in os.listdir(base_dir):
    # Verifica che il file sia di testo
    if file_name.endswith('.txt'):
        # Leggi il file di testo e crea un DataFrame
        file_path = os.path.join(base_dir, file_name)
        df = pd.read_table(file_path)
        
        # Aggiungi la colonna LesionBinary al DataFrame normalizzato
        df_normalized = df[columns_to_normalize + ['LesionBinary']].copy()
        
        # Normalizza le colonne selezionate, non la colonna Lesion binary
        df_normalized[columns_to_normalize] = scaler.fit_transform(df_normalized[columns_to_normalize])
        
        # Itera su tutti i valori unici nella colonna "Combination"
        for combination_value in df['Combination'].unique():
            # Seleziona solo le righe con il valore specificato di "Combination"
            df_filtered = df_normalized[df['Combination'] == combination_value]
            
            # Calcola le medie delle colonne
            column_means = df_filtered.mean()
            
            # Aggiungi le medie delle colonne al dizionario insieme al valore di LesionBinary
            key = (file_name, combination_value)
            mean_vectors_by_file_and_combination[key] = {'vectors': column_means[columns_to_normalize].values,
                                                         'LesionBinary': column_means['LesionBinary']}

# Stampare il numero di vettori di medie ottenuti
print("Numero di vettori di medie ottenuti:", len(mean_vectors_by_file_and_combination))

# Ora mean_vectors_by_file_and_combination contiene i vettori di medie e il valore di LesionBinary per ciascun file di testo e per ciascun valore di combination


#In questo codice, mean_vectors_by_file_and_combination è un dizionario in cui le chiavi sono tuple contenenti il nome del file di testo e il valore di "Combination",
#mentre i valori sono i vettori di medie corrispondenti. Alla fine del processo, 
#il dizionario conterrà 132 coppie di chiavi-valore, rappresentanti il nome del file di testo, il valore di "Combination" e il vettore di medie corrispondente.
#Ad esempio, se si accede a un elemento del dizionario usando una chiave come ('file1.txt', 1), 
#il valore restituito includerà i vettori di medie e il valore medio di "LesionBinary" per il file file1.txt e la combinazione 1.

Numero di vettori di medie ottenuti: 132


In [13]:
#PROVA: stampo paziente 1 con combination 1 

# Definire la chiave corrispondente al primo file di testo con combination 1
key = ('AverageValues_Voxelwise_Tau_1_P25.txt', 4)

# Accedere alle medie e al valore di LesionBinary corrispondenti usando la chiave
mean_values = mean_vectors_by_file_and_combination[key]['vectors']
lesion_binary = mean_vectors_by_file_and_combination[key]['LesionBinary']

# Stampare le medie e il valore di LesionBinary
print("Medie del primo file di testo con combination 1:")
print("Medie:", mean_values)
print("LesionBinary:", lesion_binary)


Medie del primo file di testo con combination 1:
Medie: [0.37145636 0.11717696 0.10329383 0.24668129 0.37921947 0.21692808
 0.43386809 0.5007292  0.41844835 0.22522584 0.3089613  0.69021106
 0.23625753 0.24437499 0.11789807]
LesionBinary: 1.0


In [15]:
#adesso devo usare i dati nel dizionario nell'algoritmo knn: qui prendo il 20% per il test e l'80% per laddestramento

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Estrai le feature e i target dal dizionario
#creo una lista X per le features e una lista y per i target
#X = []
#y = []
#for key, value in mean_vectors_by_file_and_combination.items():
    #X.append(value['vectors'])
    #y.append(value['LesionBinary'])

X = np.array([value['vectors'] for value in mean_vectors_by_file_and_combination.values()])
y = np.array([value['LesionBinary'] for value in mean_vectors_by_file_and_combination.values()])


# Dividi i dati in set di addestramento e di test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inizializza il classificatore k-NN
knn = KNeighborsClassifier(n_neighbors=5)

# Addestra il modello usando il set di addestramento
knn.fit(X_train, y_train)

# Valuta le prestazioni del modello sul set di test
accuracy = knn.score(X_test, y_test)
print("Accuracy:", accuracy)

from sklearn.metrics import confusion_matrix

# Calcola la matrice di confusione
conf_matrix = confusion_matrix(y_test, knn.predict(X_test))

# Estrai i valori dalla matrice di confusione
tn, fp, fn, tp = conf_matrix.ravel()

# Calcola la sensitivity e la specificity
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Sensitivity:", sensitivity)
print("Specificity:", specificity)


Accuracy: 0.7407407407407407
Sensitivity: 0.6
Specificity: 0.8235294117647058


In [7]:
#qui scegliamo i pazienti da mettere nel test e i pazienti da mettere nell'addestramento 

from sklearn.model_selection import train_test_split

# Lista dei file di testo da utilizzare per il training set (26 file)
training_files = ['AverageValues_Voxelwise_Tau_1_P1.txt', 'AverageValues_Voxelwise_Tau_1_P2.txt', 
                  'AverageValues_Voxelwise_Tau_1_P3.txt', 'AverageValues_Voxelwise_Tau_1_P4.txt', 'AverageValues_Voxelwise_Tau_1_P5.txt', 
                  'AverageValues_Voxelwise_Tau_1_P6.txt', 'AverageValues_Voxelwise_Tau_1_P7.txt', 'AverageValues_Voxelwise_Tau_1_P8.txt', 
                  'AverageValues_Voxelwise_Tau_1_P9.txt', 'AverageValues_Voxelwise_Tau_1_P10.txt', 'AverageValues_Voxelwise_Tau_1_P11.txt', 
                  'AverageValues_Voxelwise_Tau_1_P12.txt', 'AverageValues_Voxelwise_Tau_1_P13.txt', 'AverageValues_Voxelwise_Tau_1_P14.txt', 
                  'AverageValues_Voxelwise_Tau_1_P15.txt', 'AverageValues_Voxelwise_Tau_1_P16.txt', 'AverageValues_Voxelwise_Tau_1_P17.txt', 
                  'AverageValues_Voxelwise_Tau_1_P18.txt', 'AverageValues_Voxelwise_Tau_1_P19.txt', 'AverageValues_Voxelwise_Tau_1_P20.txt', 
                  'AverageValues_Voxelwise_Tau_1_P21.txt', 'AverageValues_Voxelwise_Tau_1_P22.txt', 'AverageValues_Voxelwise_Tau_1_P23.txt', 
                  'AverageValues_Voxelwise_Tau_1_P24.txt', 'AverageValues_Voxelwise_Tau_1_P25.txt', 'AverageValues_Voxelwise_Tau_1_P26.txt']

# Lista dei file di testo da utilizzare per il test set (6 file)

test_files = ['AverageValues_Voxelwise_Tau_1_P27.txt', 'AverageValues_Voxelwise_Tau_1_P28.txt', 'AverageValues_Voxelwise_Tau_1_P29.txt', 'AverageValues_Voxelwise_Tau_1_P30.txt', 'AverageValues_Voxelwise_Tau_1_P31.txt', 'AverageValues_Voxelwise_Tau_1_P32.txt', 'AverageValues_Voxelwise_Tau_1_P33.txt']

# Inizializza liste per le feature (vettori di medie) e i target (LesionBinary)
X_train1, X_test1, y_train1, y_test1 = [], [], [], []

# Costruisci il training set
for file, combination in mean_vectors_by_file_and_combination.keys():
    # Se il file è nel training set
    if file in training_files:
        # Aggiungi il vettore di medie come feature del training set
        X_train1.append(mean_vectors_by_file_and_combination[(file, combination)]['vectors'])
        # Aggiungi il valore di LesionBinary come target del training set
        y_train1.append(mean_vectors_by_file_and_combination[(file, combination)]['LesionBinary'])
    # Se il file è nel test set
    elif file in test_files:
        # Aggiungi il vettore di medie come feature del test set
        X_test1.append(mean_vectors_by_file_and_combination[(file, combination)]['vectors'])
        # Aggiungi il valore di LesionBinary come target del test set
        y_test1.append(mean_vectors_by_file_and_combination[(file, combination)]['LesionBinary'])

# Converte le liste in array NumPy
X_train1, X_test1, y_train1, y_test1 = np.array(X_train1), np.array(X_test1), np.array(y_train1), np.array(y_test1)

# Stampa le dimensioni dei set di addestramento e test
print("Dimensioni del training set (X, y):", X_train1.shape, y_train1.shape)
print("Dimensioni del test set (X, y):", X_test1.shape, y_test1.shape)
from sklearn.neighbors import KNeighborsClassifier

# Inizializza il classificatore k-NN
knn1 = KNeighborsClassifier(n_neighbors=5)

# Addestra il modello usando il set di addestramento
knn1.fit(X_train1, y_train1)

# Valuta le prestazioni del modello sul set di test
accuracy1 = knn1.score(X_test1, y_test1)
print("Accuracy:", accuracy1)

conf_matrix1 = confusion_matrix(y_test1, knn1.predict(X_test1))
tn1, fp1, fn1, tp1 = conf_matrix1.ravel()

# Calcola la sensitivity e la specificity
sensitivity1 = tp1 / (tp1 + fn1)
specificity1 = tn1 / (tn1 + fp1)

print("Sensitivity:", sensitivity1)
print("Specificity:", specificity1)



Dimensioni del training set (X, y): (104, 15) (104,)
Dimensioni del test set (X, y): (28, 15) (28,)
Accuracy: 0.6785714285714286
Sensitivity: 0.75
Specificity: 0.625


In [8]:
#nel caso precedente l'accuratezza è bassa, probabilmente perchè i dati non vengono suddivisi casualmente
#qui i file da usare nel set di addestramento e test sono scelti in modo casuale, posso però modificare la percentuale

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Lista dei file di testo disponibili
all_files = [
    'AverageValues_Voxelwise_Tau_1_P1.txt', 'AverageValues_Voxelwise_Tau_1_P2.txt', 'AverageValues_Voxelwise_Tau_1_P3.txt', 'AverageValues_Voxelwise_Tau_1_P4.txt', 'AverageValues_Voxelwise_Tau_1_P5.txt', 
    'AverageValues_Voxelwise_Tau_1_P6.txt', 'AverageValues_Voxelwise_Tau_1_P7.txt', 'AverageValues_Voxelwise_Tau_1_P8.txt', 'AverageValues_Voxelwise_Tau_1_P9.txt', 'AverageValues_Voxelwise_Tau_1_P10.txt', 
    'AverageValues_Voxelwise_Tau_1_P11.txt', 'AverageValues_Voxelwise_Tau_1_P12.txt', 'AverageValues_Voxelwise_Tau_1_P13.txt', 'AverageValues_Voxelwise_Tau_1_P14.txt', 'AverageValues_Voxelwise_Tau_1_P15.txt', 
    'AverageValues_Voxelwise_Tau_1_P16.txt', 'AverageValues_Voxelwise_Tau_1_P17.txt', 'AverageValues_Voxelwise_Tau_1_P18.txt', 'AverageValues_Voxelwise_Tau_1_P19.txt', 'AverageValues_Voxelwise_Tau_1_P20.txt', 
    'AverageValues_Voxelwise_Tau_1_P21.txt', 'AverageValues_Voxelwise_Tau_1_P22.txt', 'AverageValues_Voxelwise_Tau_1_P23.txt', 'AverageValues_Voxelwise_Tau_1_P24.txt', 'AverageValues_Voxelwise_Tau_1_P25.txt',
    'AverageValues_Voxelwise_Tau_1_P26.txt', 'AverageValues_Voxelwise_Tau_1_P27.txt', 'AverageValues_Voxelwise_Tau_1_P28.txt', 'AverageValues_Voxelwise_Tau_1_P29.txt', 'AverageValues_Voxelwise_Tau_1_P30.txt',
    'AverageValues_Voxelwise_Tau_1_P31.txt', 'AverageValues_Voxelwise_Tau_1_P32.txt', 'AverageValues_Voxelwise_Tau_1_P33.txt'
]

# Lista dei file di testo da utilizzare per il training set (26 file)
training_files2, test_files2 = train_test_split(all_files, test_size=0.15, random_state=42)

# Inizializza liste per le feature (vettori di medie) e i target (LesionBinary)
X_train2, X_test2, y_train2, y_test2 = [], [], [], []

# Costruisci il training set e il test set
for file, combination in mean_vectors_by_file_and_combination.keys():
    # Se il file è nel training set
    if file in training_files2:
        # Aggiungi il vettore di medie come feature del training set
        X_train2.append(mean_vectors_by_file_and_combination[(file, combination)]['vectors'])
        # Aggiungi il valore di LesionBinary come target del training set
        y_train2.append(mean_vectors_by_file_and_combination[(file, combination)]['LesionBinary'])
    # Se il file è nel test set
    elif file in test_files2:
        # Aggiungi il vettore di medie come feature del test set
        X_test2.append(mean_vectors_by_file_and_combination[(file, combination)]['vectors'])
        # Aggiungi il valore di LesionBinary come target del test set
        y_test2.append(mean_vectors_by_file_and_combination[(file, combination)]['LesionBinary'])

# Converte le liste in array NumPy
X_train2, X_test2, y_train2, y_test2 = np.array(X_train2), np.array(X_test2), np.array(y_train2), np.array(y_test2)

# Stampa le dimensioni dei set di addestramento e test
print("Dimensioni del training set (X, y):", X_train2.shape, y_train2.shape)
print("Dimensioni del test set (X, y):", X_test2.shape, y_test2.shape)

# Inizializza il classificatore k-NN
knn2 = KNeighborsClassifier(n_neighbors=5)

# Addestra il modello usando il set di addestramento
knn2.fit(X_train2, y_train2)

# Valuta le prestazioni del modello sul set di test
accuracy2 = knn2.score(X_test2, y_test2)
print("Accuracy: {:.5f}".format(accuracy2))

#print("Accuracy:", accuracy)

conf_matrix2 = confusion_matrix(y_test2, knn2.predict(X_test2))
tn2, fp2, fn2, tp2 = conf_matrix2.ravel()

# Calcola la sensitivity e la specificity
sensitivity2 = tp2 / (tp2 + fn2)
specificity2 = tn2 / (tn2 + fp2)

print("Sensitivity:", sensitivity2)
print("Specificity:", specificity2)




Dimensioni del training set (X, y): (112, 15) (112,)
Dimensioni del test set (X, y): (20, 15) (20,)
Accuracy: 0.60000
Sensitivity: 0.375
Specificity: 0.75


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Lista dei file di testo disponibili
all_files3 = [
    'AverageValues_Voxelwise_Tau_1_P1.txt', 'AverageValues_Voxelwise_Tau_1_P2.txt', 'AverageValues_Voxelwise_Tau_1_P3.txt', 'AverageValues_Voxelwise_Tau_1_P4.txt', 'AverageValues_Voxelwise_Tau_1_P5.txt', 
    'AverageValues_Voxelwise_Tau_1_P6.txt', 'AverageValues_Voxelwise_Tau_1_P7.txt', 'AverageValues_Voxelwise_Tau_1_P8.txt', 'AverageValues_Voxelwise_Tau_1_P9.txt', 'AverageValues_Voxelwise_Tau_1_P10.txt', 
    'AverageValues_Voxelwise_Tau_1_P11.txt', 'AverageValues_Voxelwise_Tau_1_P12.txt', 'AverageValues_Voxelwise_Tau_1_P13.txt', 'AverageValues_Voxelwise_Tau_1_P14.txt', 'AverageValues_Voxelwise_Tau_1_P15.txt', 
    'AverageValues_Voxelwise_Tau_1_P16.txt', 'AverageValues_Voxelwise_Tau_1_P17.txt', 'AverageValues_Voxelwise_Tau_1_P18.txt', 'AverageValues_Voxelwise_Tau_1_P19.txt', 'AverageValues_Voxelwise_Tau_1_P20.txt', 
    'AverageValues_Voxelwise_Tau_1_P21.txt', 'AverageValues_Voxelwise_Tau_1_P22.txt', 'AverageValues_Voxelwise_Tau_1_P23.txt', 'AverageValues_Voxelwise_Tau_1_P24.txt', 'AverageValues_Voxelwise_Tau_1_P25.txt',
    'AverageValues_Voxelwise_Tau_1_P26.txt', 'AverageValues_Voxelwise_Tau_1_P27.txt', 'AverageValues_Voxelwise_Tau_1_P28.txt', 'AverageValues_Voxelwise_Tau_1_P29.txt', 'AverageValues_Voxelwise_Tau_1_P30.txt',
    'AverageValues_Voxelwise_Tau_1_P31.txt', 'AverageValues_Voxelwise_Tau_1_P32.txt', 'AverageValues_Voxelwise_Tau_1_P33.txt'
]

# Suddivisione dei file in set di addestramento e test
training_files3, test_files3 = train_test_split(all_files3, test_size=0.15, random_state=42)

# Inizializza liste per le feature (vettori di medie) e i target (LesionBinary)
X_train3, X_test3, y_train3, y_test3 = [], [], [], []

# Costruisci il training set
for file, combination in mean_vectors_by_file_and_combination.keys():
    vectors = mean_vectors_by_file_and_combination[(file, combination)]['vectors']
    label = mean_vectors_by_file_and_combination[(file, combination)]['LesionBinary']
    if file in training_files3:
        X_train3.append(vectors)
        y_train3.append(label)
    elif file in test_files3:
        X_test3.append(vectors)
        y_test3.append(label)

# Converte le liste in array NumPy
X_train3, X_test3, y_train3, y_test3 = np.array(X_train3), np.array(X_test3), np.array(y_train3), np.array(y_test3)

# Stampa le dimensioni dei set di addestramento e test
print("Dimensioni del training set (X, y):", X_train3.shape, y_train3.shape)
print("Dimensioni del test set (X, y):", X_test3.shape, y_test3.shape)

# Inizializza il classificatore k-NN
knn3 = KNeighborsClassifier(n_neighbors=5)

# Addestra il modello usando il set di addestramento
knn3.fit(X_train3, y_train3)

# Valuta le prestazioni del modello sul set di test
accuracy3 = knn3.score(X_test3, y_test3)
print("Accuracy:", accuracy3)

conf_matrix3 = confusion_matrix(y_test3, knn3.predict(X_test3))
tn3, fp3, fn3, tp3 = conf_matrix3.ravel()

# Calcola la sensitivity e la specificity
sensitivity3 = tp3 / (tp3 + fn3)
specificity3 = tn3 / (tn3 + fp3)

print("Sensitivity:", sensitivity3)
print("Specificity:", specificity3)


Dimensioni del training set (X, y): (112, 15) (112,)
Dimensioni del test set (X, y): (20, 15) (20,)
Accuracy: 0.6
Sensitivity: 0.375
Specificity: 0.75
